# Stacked Line Legends

Showcasing the `vline_stack` defaults that mirror `pandas.DataFrame.plot()`:

- **Auto-stacker** — omit `stackers` to plot every numeric column automatically
- **Auto-legend** — column names become legend labels by default (`legend=True`)
- **Custom labels** — pass `legend_label=[...]` to override auto-derived names
- **No legend** — pass `legend=False` to suppress entirely
- **Date axis** — use any temporal column as `x` with `x_axis_type="datetime"`

In [1]:
%load_ext autoreload
%autoreload 2

# Standard library imports
from datetime import date
from datetime import timedelta

# Other imports
from bokeh.io import output_notebook
from bokeh.io import show
from bokeh.palettes import HighContrast3
import polars as pl
from xpectral import PolarsDataFrame
from xpectral.charts.theme_manager import theme
from xpectral.data import BrownianMotion

output_notebook(hide_banner=True)
theme.set("ocean")

N_PATHS = 30
N_STEPS = 252

bm = BrownianMotion(n_steps=N_STEPS, n_paths=N_PATHS, seed=42)
_df = bm.standard().with_columns([
    pl.date_range(date.today(), date.today() + timedelta(days=N_STEPS), interval="1d").alias("date"),
])
df: PolarsDataFrame = _df.select(["step", "date"] + [c for c in _df.columns if c not in ("step", "date")])

# Three-path slice used throughout the examples
paths = df.select(["step", "date", "path_0", "path_1", "path_2"])

In [2]:
fig = paths.select(["path_0", "path_1", "path_2"]).bokeh(
    title="Auto-stacker + auto-legend (0-based range index as x)",
    width=700,
    height=300,
    tools="",
    y_axis_label="W(t)",
)

fig.vline_stack(color=HighContrast3)

fig.legend.location = "top_left"
fig.legend.click_policy = "hide"

show(fig)

## 2. Date axis with auto-legend

Pass `x="date"` and `x_axis_type="datetime"` to use a temporal column as the coordinate. Legend labels still derive automatically from the stacker column names.

In [3]:
fig = paths.select(["date", "path_0", "path_1", "path_2"]).bokeh(
    title="Date axis + auto-legend",
    width=700,
    height=300,
    tools="",
    x_axis_label="Date",
    y_axis_label="W(t)",
    x_axis_type="datetime",
)

fig.vline_stack(x="date", color=HighContrast3)

fig.legend.location = "top_left"
fig.legend.click_policy = "hide"

show(fig)

## 3. Custom legend labels

Pass `legend_label=[...]` to override the auto-derived column names with any display text. The explicit list always takes precedence over `legend=True`.

In [4]:
fig = paths.select(["date", "path_0", "path_1", "path_2"]).bokeh(
    title="Custom legend labels",
    width=700,
    height=300,
    tools="",
    x_axis_label="Date",
    y_axis_label="W(t)",
    x_axis_type="datetime",
)

fig.vline_stack(
    x="date",
    color=HighContrast3,
    legend_label=["Product A", "Product B", "Product C"],
)

fig.legend.location = "top_left"
fig.legend.click_policy = "hide"

show(fig)

## 4. No legend

Pass `legend=False` to suppress the legend entirely, keeping the chart clean.

In [5]:
fig = paths.select(["date", "path_0", "path_1", "path_2"]).bokeh(
    title="No legend",
    width=700,
    height=300,
    tools="",
    x_axis_label="Date",
    y_axis_label="W(t)",
    x_axis_type="datetime",
)

fig.vline_stack(x="date", color=HighContrast3, legend=False)

show(fig)

## 1. Auto-stacker + auto-legend

Omit `stackers` and `legend_label` entirely. Every numeric column becomes a stacker, and each one is labeled with its column name — no explicit list required.